# 多模型性能对比分析

对比Baseline、KAN、DeepMLP三个模型的per-class性能
- F1 Score per Class  
- True vs Predicted Class Prevalence

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 设置字体和样式
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✅ 导入库完成")

## 1. 配置模型路径

In [ ]:
# ========================================
# 配置区域：修改这里的CSV文件路径
# ========================================

models = {
    # 修改为实际的CSV文件路径
    'Baseline': '../results/per_class_analysis_baseline/per_class_detailed_metrics.csv',
    'KAN': '../results/per_class_analysis_kan/per_class_detailed_metrics.csv',
    'DeepMLP': 'results/per_class_analysis_deep_mlp_bg_excl_20251103_200903/per_class_detailed_metrics.csv',
}

# 颜色配置
model_colors = {
    'Baseline': '#1f77b4',  # 蓝色
    'KAN': '#ff7f0e',       # 橙色  
    'DeepMLP': '#2ca02c',   # 绿色
}

# FreeSurfer标签映射Excel文件
label_mapping_file = '/Users/jannik/KAN-Brain-Single-Voxel-Segmentaion/3D_dev/training/downsampling/Freesurfer_LUT_alex_labels_jiayi (1).xlsx'

print(f"📋 配置了 {len(models)} 个模型")
for name, path in models.items():
    print(f"  - {name}: {path}")

## 2. 加载FreeSurfer标签映射

In [ ]:
# 读取Excel标签映射
try:
    label_df = pd.read_excel(label_mapping_file)
    print(f"📊 Excel列名: {label_df.columns.tolist()}")
    
    # 根据Excel文件的实际列名修改
    label_col = 'one_hot_loc_alex_label'  # class_id列
    name_col = 'tissue_name'              # 组织名称列
    
    label_mapping = dict(zip(label_df[label_col], label_df[name_col]))
    
    print(f"✅ 加载了 {len(label_mapping)} 个标签映射")
    print(f"\n示例映射:")
    for i, (k, v) in enumerate(list(label_mapping.items())[:5]):
        print(f"  {k} -> {v}")
    
except Exception as e:
    print(f"⚠️ 加载标签映射失败: {e}")
    print("将使用原始class_id作为标签")
    label_mapping = {}

## 3. 加载所有模型的CSV数据

In [ ]:
model_data = {}

for model_name, csv_path in models.items():
    print(f"\n📊 加载 {model_name}...")
    try:
        df = pd.read_csv(csv_path)
        print(f"  原始列名: {df.columns.tolist()}")
        
        # ========================================
        # 列名标准化 - 统一不同CSV格式
        # ========================================
        column_mapping = {
            'dice': 'dice_coefficient',  # baseline用dice，其他用dice_coefficient
            'class_name': 'original_class_name'  # 保留原始class_name
        }
        
        df = df.rename(columns=column_mapping)
        print(f"  标准化后列名: {df.columns.tolist()}")
        
        # 如果缺少prevalence列，从support计算（归一化）
        if 'prevalence' not in df.columns and 'support' in df.columns:
            total_support = df['support'].sum()
            if total_support > 0:
                df['prevalence'] = df['support'] / total_support
                print(f"  ✅ 从support计算prevalence")
        
        # 如果缺少predicted_prevalence，设为prevalence的副本（占位）
        if 'predicted_prevalence' not in df.columns:
            if 'prevalence' in df.columns:
                df['predicted_prevalence'] = df['prevalence']
                print(f"  ⚠️ predicted_prevalence缺失，使用prevalence作为占位")
        
        # 添加组织名称列
        if label_mapping:
            df['tissue_name'] = df['class_id'].map(label_mapping)
            df['tissue_name'] = df['tissue_name'].fillna('Class_' + df['class_id'].astype(str))
        else:
            df['tissue_name'] = 'Class_' + df['class_id'].astype(str)
        
        model_data[model_name] = df
        print(f"  ✅ 成功: {len(df)} 个类别")
        print(f"  平均F1: {df['f1_score'].mean():.4f}")
        
    except FileNotFoundError:
        print(f"  ❌ 文件未找到: {csv_path}")
        print(f"  请检查路径是否正确")
    except Exception as e:
        print(f"  ❌ 加载失败: {e}")
        import traceback
        traceback.print_exc()

print(f"\n{'='*60}")
print(f"📊 成功加载 {len(model_data)}/{len(models)} 个模型")
print(f"{'='*60}")

## 4. 数据预览

In [ ]:
if model_data:
    for model_name, df in model_data.items():
        print(f"\n{'='*60}")
        print(f"📋 {model_name} 数据预览")
        print(f"{'='*60}")
        
        # 选择要显示的列（在cell-7中已经标准化了列名）
        display_cols = ['class_id', 'tissue_name', 'f1_score', 'dice_coefficient', 
                        'precision', 'recall', 'support']
        # 只显示存在的列
        display_cols = [col for col in display_cols if col in df.columns]
        
        display(df[display_cols].head(10))
        
        print(f"\n统计信息:")
        print(f"  类别数: {len(df)}")
        print(f"  平均F1: {df['f1_score'].mean():.4f}")
        print(f"  中位F1: {df['f1_score'].median():.4f}")
        print(f"  F1≥0.8的类别: {(df['f1_score'] >= 0.8).sum()}")
        print(f"  F1≥0.6的类别: {(df['f1_score'] >= 0.6).sum()}")
        print(f"  F1=0的类别: {(df['f1_score'] == 0).sum()}")

## 5. 图表1：F1 Score per Class 对比

In [ ]:
if len(model_data) > 0:
    # 找到所有模型共同的类别
    common_classes = set(model_data[list(model_data.keys())[0]]['class_id'])
    for df in model_data.values():
        common_classes = common_classes.intersection(set(df['class_id']))
    
    common_classes = sorted(list(common_classes))
    print(f"📊 共同类别数: {len(common_classes)}")
    
    if len(common_classes) == 0:
        print("❌ 错误：没有找到所有模型共同的类别！")
        print("提示：请检查CSV文件的class_id列是否一致")
    else:
        # 计算平均F1用于排序
        avg_f1 = {}
        for class_id in common_classes:
            f1_values = [df[df['class_id'] == class_id]['f1_score'].values[0] 
                         for df in model_data.values()]
            avg_f1[class_id] = np.mean(f1_values)
        
        sorted_classes = sorted(common_classes, key=lambda x: avg_f1[x], reverse=True)
        
        # 获取组织名称
        first_df = model_data[list(model_data.keys())[0]]
        tissue_names = [first_df[first_df['class_id'] == cid]['tissue_name'].values[0] 
                       for cid in sorted_classes]
        
        # 创建图表
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 16))
        fig.suptitle('F1 Score per Class - Model Comparison', 
                     fontsize=18, fontweight='bold', y=0.995)
        
        # 上图：按平均F1排序
        x = np.arange(len(sorted_classes))
        width = 0.8 / len(model_data)
        
        for i, (model_name, df) in enumerate(model_data.items()):
            f1_scores = [df[df['class_id'] == cid]['f1_score'].values[0] 
                         for cid in sorted_classes]
            
            offset = (i - len(model_data)/2 + 0.5) * width
            ax1.bar(x + offset, f1_scores, width, 
                   label=model_name, 
                   color=model_colors[model_name], 
                   alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax1.set_xlabel('Class (sorted by average F1)', fontsize=12, fontweight='bold')
        ax1.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
        ax1.set_title('F1 Score - Sorted by Performance', fontsize=14, fontweight='bold')
        ax1.set_xticks(x)
        ax1.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in zip(tissue_names, sorted_classes)], 
                            rotation=90, ha='center', fontsize=8)
        ax1.set_ylim(0, 1)
        ax1.legend(loc='upper right', fontsize=11)
        ax1.grid(True, alpha=0.3, axis='y')
        ax1.axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
        ax1.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
        
        # 下图：按class_id排序
        sorted_by_id = sorted(common_classes)
        tissue_names_by_id = [first_df[first_df['class_id'] == cid]['tissue_name'].values[0] 
                             for cid in sorted_by_id]
        
        x2 = np.arange(len(sorted_by_id))
        
        for i, (model_name, df) in enumerate(model_data.items()):
            f1_scores = [df[df['class_id'] == cid]['f1_score'].values[0] 
                         for cid in sorted_by_id]
            
            offset = (i - len(model_data)/2 + 0.5) * width
            ax2.bar(x2 + offset, f1_scores, width, 
                   label=model_name, 
                   color=model_colors[model_name], 
                   alpha=0.8, edgecolor='black', linewidth=0.5)
        
        ax2.set_xlabel('Class (sorted by Class ID)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('F1 Score', fontsize=12, fontweight='bold')
        ax2.set_title('F1 Score - Sorted by Class ID', fontsize=14, fontweight='bold')
        ax2.set_xticks(x2)
        ax2.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in zip(tissue_names_by_id, sorted_by_id)], 
                            rotation=90, ha='center', fontsize=8)
        ax2.set_ylim(0, 1)
        ax2.legend(loc='upper right', fontsize=11)
        ax2.grid(True, alpha=0.3, axis='y')
        ax2.axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
        ax2.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
else:
    print("❌ 没有成功加载任何模型数据")

## 6. 图表2：True vs Predicted Class Prevalence 对比

In [ ]:
if len(model_data) > 0:
    # 检查是否所有模型都有prevalence数据
    has_prevalence = all('prevalence' in df.columns and 'predicted_prevalence' in df.columns 
                         for df in model_data.values())
    
    if not has_prevalence:
        print("⚠️ 部分模型缺少prevalence数据，跳过Prevalence对比图表")
        print("提示：prevalence数据通常在训练脚本的per-class分析中生成")
    else:
        n_models = len(model_data)
        fig, axes = plt.subplots(n_models, 1, figsize=(20, 8*n_models))
        
        if n_models == 1:
            axes = [axes]
        
        fig.suptitle('True vs Predicted Class Prevalence - Model Comparison', 
                     fontsize=18, fontweight='bold', y=0.995)
        
        for idx, (model_name, df) in enumerate(model_data.items()):
            ax = axes[idx]
            df_sorted = df.sort_values('class_id')
            
            x = np.arange(len(df_sorted))
            width = 0.35
            
            # 检查prevalence值是否有效
            valid_prevalence = df_sorted['prevalence'] > 0
            if valid_prevalence.sum() == 0:
                print(f"⚠️ {model_name}: 所有prevalence值为0，跳过此模型")
                continue
            
            ax.bar(x - width/2, df_sorted['prevalence'], width, 
                  label='True Prevalence', color='#1f77b4', 
                  alpha=0.8, edgecolor='black', linewidth=0.5)
            
            ax.bar(x + width/2, df_sorted['predicted_prevalence'], width, 
                  label='Predicted Prevalence', color=model_colors[model_name], 
                  alpha=0.8, edgecolor='black', linewidth=0.5)
            
            ax.set_xlabel('Class', fontsize=12, fontweight='bold')
            ax.set_ylabel('Prevalence (log scale)', fontsize=12, fontweight='bold')
            ax.set_title(f'{model_name} - True vs Predicted Prevalence', 
                        fontsize=14, fontweight='bold')
            ax.set_xticks(x)
            ax.set_xticklabels([f"{name}\n(ID:{cid})" for name, cid in 
                               zip(df_sorted['tissue_name'], df_sorted['class_id'])], 
                              rotation=90, ha='center', fontsize=8)
            ax.set_yscale('log')
            ax.legend(fontsize=11)
            ax.grid(True, alpha=0.3, axis='y')
            
            # 添加连接线
            for i in range(len(df_sorted)):
                true_val = df_sorted.iloc[i]['prevalence']
                pred_val = df_sorted.iloc[i]['predicted_prevalence']
                if true_val > 0 and pred_val > 0:
                    ax.plot([i-width/2, i+width/2], [true_val, pred_val], 
                           'k--', alpha=0.3, linewidth=0.5)
        
        plt.tight_layout()
        plt.show()
else:
    print("❌ 没有成功加载任何模型数据")

## 7. 统计汇总

In [ ]:
if len(model_data) > 0:
    summary_data = []
    
    for model_name, df in model_data.items():
        summary = {
            'Model': model_name,
            'Mean F1': df['f1_score'].mean(),
            'Median F1': df['f1_score'].median(),
            'Std F1': df['f1_score'].std(),
            'Min F1': df['f1_score'].min(),
            'Max F1': df['f1_score'].max(),
            'F1≥0.8': (df['f1_score'] >= 0.8).sum(),
            'F1≥0.6': (df['f1_score'] >= 0.6).sum(),
            'F1=0': (df['f1_score'] == 0).sum(),
            'Mean Dice': df['dice_coefficient'].mean() if 'dice_coefficient' in df.columns else np.nan,
            'Mean Precision': df['precision'].mean(),
            'Mean Recall': df['recall'].mean(),
        }
        summary_data.append(summary)
    
    summary_df = pd.DataFrame(summary_data)
    
    print("\n" + "="*100)
    print("📊 模型性能汇总表")
    print("="*100)
    display(summary_df)
    print("="*100)
    
    # 可视化汇总
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Model Performance Summary', fontsize=16, fontweight='bold')
    
    # 1. 平均F1对比
    ax = axes[0]
    bars = ax.bar(summary_df['Model'], summary_df['Mean F1'], 
                  color=[model_colors[m] for m in summary_df['Model']],
                  alpha=0.8, edgecolor='black', linewidth=1.5)
    ax.set_ylabel('Mean F1 Score', fontweight='bold')
    ax.set_title('Average F1 Score', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
               f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. F1分布箱线图
    ax = axes[1]
    f1_data = [model_data[m]['f1_score'].values for m in summary_df['Model']]
    bp = ax.boxplot(f1_data, labels=summary_df['Model'], patch_artist=True)
    for patch, model in zip(bp['boxes'], summary_df['Model']):
        patch.set_facecolor(model_colors[model])
        patch.set_alpha(0.8)
    ax.set_ylabel('F1 Score', fontweight='bold')
    ax.set_title('F1 Score Distribution', fontweight='bold')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 3. 性能等级分布
    ax = axes[2]
    categories = ['Excellent\n(F1≥0.8)', 'Good\n(F1≥0.6)', 'Failed\n(F1=0)']
    x = np.arange(len(categories))
    width = 0.8 / len(model_data)
    
    for i, (model_name, _) in enumerate(model_data.items()):
        counts = [
            summary_df[summary_df['Model'] == model_name]['F1≥0.8'].values[0],
            summary_df[summary_df['Model'] == model_name]['F1≥0.6'].values[0],
            summary_df[summary_df['Model'] == model_name]['F1=0'].values[0],
        ]
        offset = (i - len(model_data)/2 + 0.5) * width
        ax.bar(x + offset, counts, width, 
              label=model_name, 
              color=model_colors[model_name],
              alpha=0.8, edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel('Number of Classes', fontweight='bold')
    ax.set_title('Performance Category Distribution', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("❌ 没有成功加载任何模型数据")

## 8. 完成

✅ 所有图表已生成！

### 使用说明：

1. 在第1步配置中，修改三个模型的CSV文件路径
2. 点击 `Cell` → `Run All` 运行所有单元格
3. 查看生成的对比图表